# EDSS 남은 식별자 공백 처리 검산

## TL;DR

취업통계 2023–2024년 46,962행 전체를 형식 단절로 기존 OpenID 종단 패널에서 제외한 결정을 재검산한다. 원본과 안전 파생 자료는 독립 참고용으로 보존하며, 고위험 패널 4개의 수동 검토 잔여도 0개다.

## Context & Methods

- 취업통계 2010–2022년의 OpenID별 학과·단과대 서명을 2023–2024년 학교·연도별 서명과 완전 일치 비교한다.
- 서명 크기가 3 미만이거나 복수 OpenID와 일치하면 미해결로 둔다.
- 단일 후보도 같은 연도의 0101 시도·본분교 맥락을 추가 확인한다.
- 파생 파일에는 2023–2024년 공개 집계 필드와 출처 추적 필드만 둔다.
- 고위험 패널의 미연결 키는 기준기간 경계와 내부 공백으로 분리한다.
- 승인된 내부 공백 판정은 기대 행 수·공식 근거·비대입 처리 규칙을 검증한 뒤 별도 판정층으로 적용한다.
- 취업통계 범위 결정은 공식 교차표 감사와 추론 적용 감사의 행 수를 교차검증하고, 2023–2024년 전체를 기존 종단 패널에서 제외한다.

## Data

In [1]:
from pathlib import Path
from collections import Counter
import csv, gzip, hashlib, json
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / 'data/metadata').exists():
    ROOT = ROOT.parent
assert (ROOT / 'data/metadata').exists(), 'repository root not found'
summary_path = ROOT / 'data/metadata/edss_remaining_identity_gap_resolution.json'
candidate_path = ROOT / 'data/metadata/edss_employment_2023_2024_open_id_candidates.csv'
review_path = ROOT / 'data/metadata/edss_high_orphan_panel_review.csv'
decision_path = ROOT / 'data/metadata/edss_high_orphan_manual_decisions.csv'
scope_decision_path = ROOT / 'data/metadata/edss_employment_schema_break_decision.csv'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
derived_path = ROOT / summary['outputs']['derived_employment']['path']
with candidate_path.open(encoding='utf-8-sig', newline='') as handle:
    candidates = list(csv.DictReader(handle))
with review_path.open(encoding='utf-8-sig', newline='') as handle:
    reviews = list(csv.DictReader(handle))
with decision_path.open(encoding='utf-8-sig', newline='') as handle:
    manual_decisions = list(csv.DictReader(handle))
with scope_decision_path.open(encoding='utf-8-sig', newline='') as handle:
    scope_decisions = list(csv.DictReader(handle))
with gzip.open(derived_path, 'rt', encoding='utf-8', newline='') as handle:
    reader = csv.DictReader(handle)
    derived_fields = reader.fieldnames
    derived_rows = sum(1 for _ in reader)
derived_sha256 = hashlib.sha256(derived_path.read_bytes()).hexdigest()
display(Markdown(f'입력 요약 1개, 후보 상태 {len(candidates):,}개, 고위험 패널 {len(reviews)}개, 패널 수동 판정 {len(manual_decisions)}개, 취업 범위 판정 {len(scope_decisions)}개를 읽었다.'))

입력 요약 1개, 후보 상태 3,281개, 고위험 패널 4개, 패널 수동 판정 1개, 취업 범위 판정 1개를 읽었다.

## Results

In [2]:
employment = summary['employment']
high = summary['high_orphan_panels']
status_counts = Counter(row['resolution_status'] for row in candidates)
forbidden_individual_fields = {
    '성명', '주민등록번호', '외국인등록번호', '생년월일', '성별', '전화번호', '이메일', '주소'
}
assert derived_rows == 46_962
assert derived_rows == summary['outputs']['derived_employment']['row_count']
assert derived_sha256 == summary['outputs']['derived_employment']['sha256']
assert summary['status'] == 'complete_with_scope_exclusion'
assert employment['status'] == 'complete_with_scope_exclusion'
assert employment['canonical_open_id_imputed_row_count'] == 0
assert employment['decision_classification'] == 'schema_break_excluded'
assert employment['excluded_years'] == ['2023', '2024']
assert employment['scope_excluded_row_count'] == 46_962
assert employment['scope_excluded_school_year_identity_count'] == 3_281
assert employment['reference_only_inferred_open_id_row_count'] == 13_920
assert employment['unresolved_open_id_row_count_at_exclusion'] == 33_042
assert employment['legacy_panel_eligible_row_count'] == 0
assert employment['raw_and_derived_records_preserved'] is True
assert len(scope_decisions) == 1
assert scope_decisions[0]['decision_status'] == 'approved'
assert '개방ID' not in derived_fields
assert forbidden_individual_fields.isdisjoint(derived_fields)
assert sum(status_counts.values()) == 3_281
assert status_counts['candidate_signature_context_confirmed'] == 30
assert status_counts['candidate_0101_context_conflict'] == 2
rows = [
    ('파생 취업 집계 행', f'{derived_rows:,}'),
    ('학교·연도 상태', f'{len(candidates):,}'),
    ('0101 맥락 일치 후보', f"{status_counts['candidate_signature_context_confirmed']:,}"),
    ('0101 맥락 충돌 후보', f"{status_counts['candidate_0101_context_conflict']:,}"),
    ('형식 단절 제외 행', f"{employment['scope_excluded_row_count']:,}"),
    ('기존 종단 패널 적격 행', f"{employment['legacy_panel_eligible_row_count']:,}"),
    ('참고용 추론 ID 적용 기록', f"{employment['reference_only_inferred_open_id_row_count']:,}"),
    ('제외 시점 미연결 행', f"{employment['unresolved_open_id_row_count_at_exclusion']:,}"),
    ('정식 개방ID 대입', '0'),
    ('개인 필드 검출', '0'),
]
table = '| 검산 항목 | 결과 |\n|---|---:|\n' + '\n'.join(f'| {k} | {v} |' for k, v in rows)
display(Markdown(table))

| 검산 항목 | 결과 |
|---|---:|
| 파생 취업 집계 행 | 46,962 |
| 학교·연도 상태 | 3,281 |
| 0101 맥락 일치 후보 | 30 |
| 0101 맥락 충돌 후보 | 2 |
| 형식 단절 제외 행 | 46,962 |
| 기존 종단 패널 적격 행 | 0 |
| 참고용 추론 ID 적용 기록 | 13,920 |
| 제외 시점 미연결 행 | 33,042 |
| 정식 개방ID 대입 | 0 |
| 개인 필드 검출 | 0 |

In [3]:
assert len(reviews) == 4
assert high['explained_temporal_boundary_panel_count'] == 3
assert high['status'] == 'complete'
assert high['manual_review_required_panel_count'] == 0
assert high['manual_review_keys'] == []
assert high['manually_resolved_panel_count'] == 1
assert high['manually_resolved_key_count'] == 1
assert len(manual_decisions) == 1
decision = manual_decisions[0]
assert decision['catalog_code'] == '1209'
assert decision['year'] == '2019'
assert decision['open_id'] == '5831784427'
assert int(decision['expected_row_count']) == 16
assert decision['decision_status'] == 'approved'
assert decision['recommended_handling'].startswith('retain_same_open_id')
table = '| 코드 | 데이터셋 | 판정 | 경계 키 | 내부 공백 키 | 수동 확정 키·행 |\n|---|---|---|---:|---:|---:|\n'
table += '\n'.join(
    f"| {r['catalog_code']} | {r['dataset']} | {r['review_disposition']} | {r['boundary_key_count']} | {r['internal_gap_key_count']} | {r['manually_resolved_key_count']}·{r['manually_resolved_row_count']} |"
    for r in reviews
)
display(Markdown(table))

| 코드 | 데이터셋 | 판정 | 경계 키 | 내부 공백 키 | 수동 확정 키·행 |
|---|---|---|---:|---:|---:|
| 0202 | 대학입학전형기본계획_전문대학 | explained_temporal_boundary | 26 | 0 | 0·0 |
| 0204 | 대학입학전형시행계획_전문대학 | explained_temporal_boundary | 12 | 0 | 0·0 |
| 1102 | 도서관예산현황 | explained_temporal_boundary | 35 | 0 | 0·0 |
| 1209 | 법인임원현황 | explained_manual_identity_scope_decision | 66 | 1 | 1·16 |

## Takeaways

- 2023–2024년 취업통계 46,962행·3,281개 학교연도 전체는 `schema_break_excluded`로 기존 OpenID 종단 패널에서 제외됐다.
- 원본과 안전 파생 자료는 삭제하지 않고 2023–2024년 독립 참고 자료로 보존한다. 13,920개 추론 적용 행과 33,042개 미연결 행 모두 종단 패널 적격 행이 아니다.
- 30개 학교·연도 후보는 학과서명과 0101 맥락이 모두 맞지만 공식 교차표가 아니므로 확정 ID가 아니다.
- 0202·0204·1102의 고위험 미연결은 모두 기준기간 경계로 설명된다.
- 1209의 2019년 내부 공백 1개·16행은 남인천캠퍼스의 학위과정 범위 차이로 확정했다. 동일 OpenID와 원본 행을 보존하며 대입·보간하지 않는다.
- 고위험 패널 4개의 수동 검토 잔여는 0개다. 전체 요약은 승인된 범위 제외를 반영해 `complete_with_scope_exclusion`으로 종결한다.